# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [2]:
# import pypyodbc
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.colors import to_hex

# 2.0 Import spot position and size QA data

In [3]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)
df.head(2)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

In [5]:
df.value_counts("MachineName"), df.value_counts("Device"), df.value_counts("Energy")

(MachineName
 Gantry 3    10576
 Gantry 1    10472
 Gantry 4    10357
 Gantry 2     9767
 Name: count, dtype: int64,
 Device
 XRV-3000    31815
 XRV-4000     9357
 Name: count, dtype: int64,
 Energy
 150    8250
 240    8243
 200    8233
 100    8232
 70     8214
 Name: count, dtype: int64)

# 4.0 filtering data

In [6]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

## calculate abs shift

In [7]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

In [8]:
print(sub_df.head(2))

                ADate MachineName  Energy    Device  Gantry Angle  \
0 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   
1 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   

            Spot     x-pos     y-pos  px_pos  py_pos  abs_xpos  abs_ypos  
0  Bottom-Centre   -0.2357  124.9224       0     125   -0.2357   -0.0776  
1    Bottom-Left -125.0153  125.4501    -125     125   -0.0153    0.4501  


In [9]:

def plotly_spot_position(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        pos = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]


    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot', 
        color='Spot',        # hue
         color_discrete_sequence= px.colors.qualitative.T10,
        title=f'{gantry} - absolute shift- {pos}',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 


# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)


In [10]:
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

## plot another device

In [11]:
plotly_spot_position(sub_df, "abs_xpos", "Gantry 4", "XRV-3000", 70, 0, 24)

In [12]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)
selected_df = sub_df[(df["MachineName"]=="Gantry 2") & (df["Device"] == "XRV-3000") & (df['ADate'] >= start_date)].copy()
# Calculate average abs_xpos per adate and energy
selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["abs_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,avg_abs_pos
37618,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Bottom-Centre,-0.2267,125.1980,0,125,-0.2267,0.1980,-0.300178
37619,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Bottom-Left,-125.3861,125.1288,-125,125,-0.3861,0.1288,-0.300178
37620,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Bottom-Right,125.0294,125.0969,125,125,0.0294,0.0969,-0.300178
37621,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Centre,-0.3810,0.1202,0,0,-0.3810,0.1202,-0.300178
37622,2025-10-08 16:45:32,Gantry 2,70,XRV-3000,0,Left,-125.5507,0.4636,-125,0,-0.5507,0.4636,-0.300178


In [13]:

def plotly_ave_spot_position(df, parameter, gantry, device,  n_months):
    """ plot average spot position across all spot positions with the same adate and energy
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        paramter = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()


    # Calculate average abs_xpos per adate and energy
    selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])[parameter].transform('mean')

    # want to displace energy as discrete colour not spectrum
    selected_df['Energy'] = df['Energy'].astype(int).astype(str)

    

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y='avg_abs_pos',
        symbol='Gantry Angle', 
        color='Energy',        # hue
        title=f'average {parameter} across all spot positions with the same adate and energy',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
       # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 




In [14]:
plotly_ave_spot_position(sub_df, "abs_xpos", "Gantry 1", "XRV-3000",  24)

# 5.0 Generalise plotting function - plotly_ave_spot_position()

## Function: plot_ave_spot_position_by_en_ga()

In [15]:

def plot_ave_spot_position_by_en_ga(df, option, parameter, gantry, device, energy, n_months, tolerance):
    """ plot average spot position across all spot positions with the same adate and energy
        df = dataframe
        option (str) = "spot_position" | "fwhm" | "spot_symmetry"
        gantry = "Gantry 1", "Gantry 2", 
        paramter = 
                    spot position option:  'abs_xpos', 'abs_ypos',  'rel_xpos', 'rel_ypos'
                    fwhm option: 'hor_fwhm', 'vert_fwhm', 'bltr_fwhm', 'tlbr_fwhm', "ave_fwhm"
                    spot_symmetry option : 'hor_rt_gradient', 'hor_lt_gradient', 
        device = "XRV-3000", "XRV-4000"
        energy = int
        n_month = int
        tolerence (float) = 2 (spot pos, abs) | fwhm from tps or baseline | None (gradient_ratio)

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    
    

    if "spot_position" in option or "spot_symmetry" in option:
        colour_column = "Energy"

        selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) ].copy()

        # Calculate average abs_xpos per adate and energy
        selected_df[f'avg_{parameter}'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])[parameter].transform('mean')

        # define title
        energies = pd.unique(df['Energy']).tolist()
        title = f"{gantry} - {device} -- Average {parameter} by Date and Energy {energies} MeV"

    elif "fwhm" in option:

        colour_column = "Gantry Angle"
        selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df['Energy'] == energy)].copy()

        # Calculate average abs_xpos per adate and energy
        selected_df[f'avg_{parameter}'] = selected_df.groupby(['ADate', "Gantry Angle"])[parameter].transform('mean')

        # define title
        energies = pd.unique(df['Energy']).tolist()
        title = f"{gantry} - {device} -- Average {parameter} by Date and Energy {energy} MeV"

   

    # want to displace energy as discrete colour not spectrum
    selected_df['Energy'] = selected_df['Energy'].astype(int).astype(str)
    # change gantry angle column datatype from integer to string
    selected_df["Gantry Angle"] = selected_df["Gantry Angle"].astype(str)

    y_axis_name = {'abs_xpos' : "Average Absolute X Position (mm)", 
                    'abs_ypos': "Average Absolute Y Position (mm)",
                    'rel_xpos': "Average Relative X Position (mm)", 
                    'rel_ypos' : "Average Relative Y Position (mm)", 
                    'hor_fwhm' : "Average Horizontal FWHM (mm)",
                    'vert_fwhm' : "Average Vertical FWHM (mm)", 
                    'bltr_fwhm' : "Average BLTR FWHM (mm)",
                    'tlbr_fwhm' : "Average TLBR FWHM (mm)", 
                    'ave_fwhm' : "Average ave_FWHM (mm)", 
                    'gr_hor' : "Average Horizontal Gradient Ratio (RT/LT)", 
                    'gr_vert' : "Average Vertical Gradient Ratio (RT/LT)", 
                    'gr_bltr' : "Average BLTR Gradient Ratio (RT/LT)",
                    'gr_tlbr' : "Average TLBR Gradient Ratio (RT/LT)"}

    # set colour
    palette = sns.color_palette("hls", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=f'avg_{parameter}',
        symbol='Gantry Angle', 
        color=colour_column,        # hue
        color_discrete_sequence= palette_hex,
        title = title,
        labels= {'ADate': 'Date',
                f'avg_{parameter}': y_axis_name[parameter],
                }, #ename axis titles, legend titles, and hover labels
        hover_data={'ADate': '|%Y-%m-%d'},
        height=500,
        opacity = 0.5
    )

    # Add tolerance bands +/- 2
    if "spot_position" in option:
        fig.add_hline(y=tolerance, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
        fig.add_hline(y=-tolerance, line_dash="dash", line_color="grey")
        
    
    elif "fwhm" in option:
        fig.add_hline(y=0.9*tolerance, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
        fig.add_hline(y=1.1*tolerance, line_dash="dash", line_color="grey")


    
    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size 
                                line=dict(width=2)),     # outline width
                                line=dict(width=1                # thinner connecting lines
                                ),
                    hovertemplate= "Date: %{x|%Y-%m-%d}<br>" +
                                    f"{parameter}: %{{y}}<br>" +
                                    "Spot: %{fullData.name}<extra></extra>")

    # Show plot
    fig.show()


    return 




### spot position data - calculate absolute and relative shift

In [16]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

# create a dict with ADate as the key and abs x, y pos as the value
centre_spot_df = sub_df[sub_df["Spot"] == "Centre"]
centre_abs_xpos = (
                    centre_spot_df
                    .set_index(["ADate", "Energy"])["abs_xpos"]
                    .to_dict()
                    )

centre_abs_ypos = (
                    centre_spot_df
                    .set_index(["ADate", "Energy"])["abs_ypos"]
                    .to_dict()
                    )


# Map using (ADate, Energy)
sub_df["centre_abs_xpos"] = (
                                sub_df[["ADate", "Energy"]]
                                .apply(tuple, axis=1)
                                .map(centre_abs_xpos)
                                )

sub_df["centre_abs_ypos"] = (
                                sub_df[["ADate", "Energy"]]
                                .apply(tuple, axis=1)
                                .map(centre_abs_ypos)
                                )

sub_df["rel_xpos"] = sub_df['abs_xpos'] - sub_df["centre_abs_xpos"]
sub_df["rel_ypos"] = sub_df['abs_ypos'] - sub_df["centre_abs_ypos"]



### spot position plots
- abs_xpos, abs_ypos | tolerance = 2
- rel_xpos, rel_ypos | tolerance = 1

In [25]:
# ascending order
sub_df.sort_values(["Energy", "Gantry Angle"], ascending=[True, True],inplace=True)

In [18]:
# plot_ave_spot_position_by_en_ga(df, option, parameter, gantry, device, energy,  n_months, tolerance)
plot_ave_spot_position_by_en_ga(sub_df, "spot_position", "abs_xpos", "Gantry 1", "XRV-3000", 150, 24, 2)

### FWHM data

In [26]:
# load fwhm reference dataset
# read data 
ref_fwhm_df = pd.read_excel(r"../data/xlsx_exported_from_access/ref/gantry_specific_baseline_2026_05_29.xlsx", sheet_name = "summary", header =0, usecols="A:E")
ref_fwhm_df["fwhm"] = ref_fwhm_df["fwhm"].round(4)


# commissioning data/ or the first annual QA data 
baseline = ref_fwhm_df[ref_fwhm_df["data_type"] =="baseline"]
baseline_ref = {}
for _, row in baseline.iterrows():
    baseline_ref.setdefault(row["gantry"], {}).setdefault(row["gantry_angle"], {})[row["energy"]] = row["fwhm"]

# tps data
tps = ref_fwhm_df[ref_fwhm_df["data_type"] =="tps"]
tps_ref = {}
for _, row in tps.iterrows():
    tps_ref.setdefault(row["gantry"], {}).setdefault(row["gantry_angle"], {})[row["energy"]] = row["fwhm"]

In [27]:
fwhm_df = df[['ADate', 'MachineName', 'Energy', 'Device', 'Gantry Angle', 'Spot', 'hor_fwhm', 'vert_fwhm','bltr_fwhm', 'tlbr_fwhm']].copy()
fwhm_df["ave_fwhm"] = fwhm_df[['hor_fwhm', 'vert_fwhm','bltr_fwhm', 'tlbr_fwhm']].mean(axis=1)

### 

#### fwhm plots
- 'hor_fwhm', 'vert_fwhm', 'bltr_fwhm', 'tlbr_fwhm', 'ave_fwhw' 
- tps_ref[1][90][e] , e (int) = 70, 100, 150, 200,240
- baseline_ref[g][ga][e], g (int) = 1, 2, 3, 4 | ga (int) = 0, 90, 180, 240
- need to write a sentence - reference fwhm = average of horizontal and vertical fwhm

In [28]:
tps_ref

{1: {90: {70: 13.5625, 100: 11.1772, 150: 9.0818, 200: 8.1719, 240: 7.4099}}}

In [22]:
# plot_ave_spot_position_by_en_ga(df, option, parameter, gantry, device, n_months, tolerance)
plot_ave_spot_position_by_en_ga(fwhm_df, "fwhm", "hor_fwhm", "Gantry 2", "XRV-3000", 240, 24, tps_ref[1][90][240])

### Gradient ratio data

In [23]:
gr_df = df[['ADate', 'MachineName', 'Energy', 'Device', 'Gantry Angle', 'Spot', 'hor_rt_gradient', 'hor_lt_gradient', 
       'vert_rt_gradient', 'vert_lt_gradient', 'bltr_rt_gradient','bltr_lt_gradient', 'tlbr_rt_gradient', 'tlbr_lt_gradient']].copy()

# gradient ratio (-1 is when the spot is perfectly symmetrical)
gr_df['gr_hor'] = gr_df["hor_rt_gradient"] / gr_df["hor_lt_gradient"] # horizontal gradient ratio
gr_df['gr_vert']  = gr_df["vert_rt_gradient"] / gr_df["vert_lt_gradient"] # vertical gradient ratio
gr_df['gr_bltr']  = gr_df["bltr_rt_gradient"] / gr_df["bltr_lt_gradient"] # bottom-left to top-right gradient ratio
gr_df['gr_tlbr']  = gr_df["tlbr_rt_gradient"] / gr_df["tlbr_lt_gradient"] # top-left to bottom-right gradient ratio

#### Gradient ratio plots
- 'gr_hor', 'gr_vert', 'gr_bltr','gr_tlbr'
- tolerance = None

In [24]:
plot_ave_spot_position_by_en_ga(gr_df, "spot_symmetry", "gr_hor", "Gantry 2", "XRV-3000", 240, 24, None)